# Watch — significant events and pending cases

Step 1 of the pipeline: **which events deserve treatment**. This notebook is
meant to be run regularly, and it produces the raw material for the weekly
digest (post type 1).

It answers three questions:

1. What significant natural catastrophes happened over the window?
2. Which of them are new this week, and which have simply been running for months?
3. For events whose footprint cannot be mapped yet, **when will the satellite pass?**

No loss figure appears anywhere in this notebook, and none should. The weekly
digest is deliberately the format that carries no number that can be wrong.

Sources: GDACS for the events, the Copernicus Data Space catalogue for the
satellite passes. Both are open and need no account.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import natcat

# ---- Settings -------------------------------------------------------------
WINDOW_DAYS = 7                      # length of the window
ALERT_LEVELS = ("Orange", "Red")     # severity filter; see the note below
AS_OF = None                         # None = today. Set a date to rebuild an
                                     # earlier digest exactly as it stood then.
# ---------------------------------------------------------------------------

# UTC, because that is the clock GDACS and the satellite catalogue run on.
end = AS_OF or datetime.now(timezone.utc).date()
start = end - timedelta(days=WINDOW_DAYS)
print(f"Window: {start} to {end}  ({WINDOW_DAYS} days)")
print(f"Alert levels: {' and '.join(ALERT_LEVELS)}")

Window: 2026-08-18 to 2026-08-25  (7 days)
Alert levels: Orange and Red


## 1. Significant events

GDACS scores every event it detects as **Green**, **Orange** or **Red**. That
score is the severity filter: Orange and Red are the events worth covering,
Green is the constant background of a planet where something is always
happening.

Two things about this feed are worth knowing, because both can quietly produce
a wrong digest.

**The feed caps every answer at 100 events and offers no way to page past it.**
A query wide enough to hit the cap loses the rest in silence. `gdacs_events`
therefore sends one query per peril, so each peril gets its own budget of 100,
and raises a visible warning if any single peril still comes back full. Adding
`"Green"` to `ALERT_LEVELS` will hit that cap within a single week and push the
serious events out of the list — it is not a way to see more.

**The window filters on overlap, not on start date.** A drought that began last
November is returned in this week's window because it is still running. Each
event therefore carries a status: *new* means it started inside the window,
*continuing* means it was already under way. Without that split, five
long-running droughts would dominate a week in which nothing much happened.

In [2]:
events = natcat.gdacs_events(days=WINDOW_DAYS, alert_levels=ALERT_LEVELS, end=AS_OF)

table = pd.DataFrame(
    [
        {
            "status": "new" if e["is_new"] else "continuing",
            "peril": natcat.GDACS_PERILS.get(e["event_type"], e["event_type"]),
            "alert": e["alert_level"],
            "country": e["country"][:40],
            "started": e["from_date"].date(),
            "severity": e["severity_text"],
            "event_id": e["event_id"],
        }
        for e in events
    ]
)

print(f"{len(events)} events at {' / '.join(ALERT_LEVELS)} level")
table if not table.empty else "Nothing at this severity in the window."

8 events at Orange / Red level


,status,peril,alert,country,started,severity,event_id
0,new,Earthquake,Orange,Afghanistan,2026-08-25,"Magnitude 4.9M, Depth:10km",1561626
1,continuing,Wildfire,Orange,Serbia,2026-08-05,Orange impact for forestfire in 18310 ha,1030252
2,continuing,Flood,Orange,China,2026-07-31,Magnitude 0,1104081
3,continuing,Drought,Orange,"Democratic Republic of Congo, Kenya, Tan",2026-05-21,Medium impact for agricultural drought in 1599...,1027465
4,continuing,Drought,Orange,"Belize, Guatemala, Honduras, Mexico, Nic",2026-05-11,Medium impact for agricultural drought in 3435...,1027449
5,continuing,Drought,Orange,"Eritrea, Ethiopia, Kenya, Somalia",2026-04-21,Medium impact for agricultural drought in 5701...,1027450
6,continuing,Drought,Orange,"Albania, Austria, Bosnia & Herzegovina,",2025-12-11,Medium impact for agricultural drought in 1487...,1018332
7,continuing,Drought,Orange,Madagascar,2025-11-21,Medium impact for agricultural drought in 3418...,1018431


In [3]:
if table.empty:
    print("Nothing at this severity in the window.")
else:
    new = table[table["status"] == "new"]
    continuing = table[table["status"] == "continuing"]

    print(f"New this window ({len(new)}):")
    display(new.drop(columns="status") if len(new) else "  none")

    print(f"\nStill running from before ({len(continuing)}):")
    display(continuing.drop(columns="status") if len(continuing) else "  none")

    print("\nBy peril and alert level:")
    display(pd.crosstab(table["peril"], table["alert"]))

New this window (1):


,peril,alert,country,started,severity,event_id
0,Earthquake,Orange,Afghanistan,2026-08-25,"Magnitude 4.9M, Depth:10km",1561626



Still running from before (7):


,peril,alert,country,started,severity,event_id
1,Wildfire,Orange,Serbia,2026-08-05,Orange impact for forestfire in 18310 ha,1030252
2,Flood,Orange,China,2026-07-31,Magnitude 0,1104081
3,Drought,Orange,"Democratic Republic of Congo, Kenya, Tan",2026-05-21,Medium impact for agricultural drought in 1599...,1027465
4,Drought,Orange,"Belize, Guatemala, Honduras, Mexico, Nic",2026-05-11,Medium impact for agricultural drought in 3435...,1027449
5,Drought,Orange,"Eritrea, Ethiopia, Kenya, Somalia",2026-04-21,Medium impact for agricultural drought in 5701...,1027450
6,Drought,Orange,"Albania, Austria, Bosnia & Herzegovina,",2025-12-11,Medium impact for agricultural drought in 1487...,1018332
7,Drought,Orange,Madagascar,2025-11-21,Medium impact for agricultural drought in 3418...,1018431



By peril and alert level:


alert,Orange
peril,
Drought,5
Earthquake,1
Flood,1
Wildfire,1


## 2. Pending cases — when the satellite passes

A flood only becomes mappable once a radar satellite has flown over it. Until
then there is no footprint, no exposed value and no loss: the chain simply
cannot start. That is a normal state of a reactive system, not a failure.

The expected pass date is the differentiating element of this whole watch.
Alerts are published by everyone; **when the picture will exist** is published
by nobody.

Two details make the estimate trustworthy rather than merely plausible.

**It measures coverage, not proximity.** One Sentinel-1 acquisition is a strip
about 250 km wide, so a pass can clip a corner of an area and leave the flooded
part unseen. All acquisitions of the same day are merged and measured against
the area of interest; a day only counts as a pass if it covered at least 90% of
it. Ask for a very large area and no single day will ever reach that threshold —
the function says so out loud rather than returning nothing.

**It is empirical.** The revisit interval is not taken from an orbit model or a
published acquisition plan, both of which change. It is measured from what the
satellite actually did over this exact box in the last month, then projected
forward. Over Europe the constellation now covers most areas almost daily; over
the tropics the interval is the classic 12 days. The estimate reflects whichever
is true for the area in question.

Only floods are checked. Wildfire extent comes from FIRMS, which is daily;
earthquake intensity from a USGS ShakeMap, published within hours; cyclone
tracks from the forecast itself. None of them queue behind an orbit.

In [4]:
# One or two network calls per flood event, so this is the slow cell.
cases = natcat.pending_cases(events, end=AS_OF)

floods = [e for e in events if e["event_type"] == "FL"]
print(f"{len(floods)} flood events in the window, {len(cases)} still waiting on a pass")

pending = pd.DataFrame(
    [
        {
            "country": c["event"]["country"][:40],
            "alert": c["event"]["alert_level"],
            "started": c["event"]["from_date"].date(),
            "last pass (pre-event)": c["last_pass"],
            "revisit (days)": c["cycle_days"],
            "next pass (est.)": c["next_pass"],
            "GFM extent expected": (
                c["gfm_expected"].strftime("%Y-%m-%d %H:%M") if c["gfm_expected"] else None
            ),
            "event_id": c["event"]["event_id"],
        }
        for c in cases
    ]
)
pending if not pending.empty else (
    "Nothing pending: every flood in the window has already been observed."
)

1 flood events in the window, 0 still waiting on a pass


'Nothing pending: every flood in the window has already been observed.'

The **GFM extent expected** column adds the measured product latency to the
pass: on the Emilia-Romagna reference case the first Global Flood Monitoring
scene appeared about 19 hours after the acquisition. It is an expectation, not
a guarantee.

A flood that does *not* appear in this table has already been observed, which
means its extent can be computed now.

## 3. Checking one area directly

Useful when an area of interest is drawn by hand rather than taken from GDACS —
for instance a Copernicus EMS activation. The two reference cases of this
project are below.

In [5]:
AREAS = {
    "EMSR664 Emilia-Romagna": (11.55, 44.05, 12.30, 44.45),
    "EMSR926 Latvia": (21.40, 56.60, 22.60, 57.30),
}

pd.DataFrame(
    [
        {"area": name, **{k: v for k, v in
                          natcat.next_satellite_pass(aoi, end=AS_OF).items()
                          if k != "passes"}}
        for name, aoi in AREAS.items()
    ]
).set_index("area")

,last_pass,next_pass,cycle_days,n_passes,best_coverage
area,,,,,
EMSR664 Emilia-Romagna,2026-08-24,2026-08-26,1,10,1.0
EMSR926 Latvia,2026-08-24,2026-08-26,1,15,1.0


## 4. Raw material for the digest

Facts only, in the order the digest uses them. The wording, the fixed
structure and the mandatory phrasings are Phase 4's business, not this
notebook's — nothing below is publishable text.

In [6]:
lines = [f"Window: {start} to {end}"]

for status in ("new", "continuing"):
    group = [e for e in events if ("new" if e["is_new"] else "continuing") == status]
    lines.append(f"\n{status.upper()} ({len(group)}):")
    for e in group:
        peril = natcat.GDACS_PERILS.get(e["event_type"], e["event_type"])
        lines.append(
            f"  {peril} | {e['country']} | {e['alert_level']} | "
            f"from {e['from_date'].date()} | {e['severity_text']} | "
            f"GDACS {e['event_id']}"
        )

lines.append(f"\nPENDING SATELLITE PASS ({len(cases)}):")
for c in cases:
    lines.append(
        f"  Flood | {c['event']['country']} | started {c['event']['from_date'].date()} | "
        f"next pass est. {c['next_pass']} | revisit {c['cycle_days']} d"
    )

print("\n".join(lines))

Window: 2026-08-18 to 2026-08-25

NEW (1):
  Earthquake | Afghanistan | Orange | from 2026-08-25 | Magnitude 4.9M, Depth:10km | GDACS 1561626

CONTINUING (7):
  Wildfire | Serbia | Orange | from 2026-08-05 | Orange impact for forestfire in 18310 ha | GDACS 1030252
  Flood | China | Orange | from 2026-07-31 | Magnitude 0  | GDACS 1104081
  Drought | Democratic Republic of Congo, Kenya, Tanzania, Uganda | Orange | from 2026-05-21 | Medium impact for agricultural drought in 159974 km2 | GDACS 1027465
  Drought | Belize, Guatemala, Honduras, Mexico, Nicaragua, El Salvador | Orange | from 2026-05-11 | Medium impact for agricultural drought in 343582 km2 | GDACS 1027449
  Drought | Eritrea, Ethiopia, Kenya, Somalia | Orange | from 2026-04-21 | Medium impact for agricultural drought in 570110 km2 | GDACS 1027450
  Drought | Albania, Austria, Bosnia & Herzegovina, Belgium, Bulgaria, Belarus, Switzerland, Czech Republic, Germany, Denmark, Spain, France, Croatia, Hungary, Ireland, Italy, Liechte

## What this notebook does not do

- **No loss figure, and no exposed value.** Those belong to the event
  notebooks under `notebooks/events/`, which run the full chain on one
  chosen event.
- **No editorial text.** Phase 4 owns the vocabulary, the structure and the
  locked phrasings.
- **No graphics.** Phase 2 owns the visual identity, the LinkedIn formats and
  the fixed blocks.

## Reading the result

The list is a shortlist, not a queue. Choosing which event to cover stays a
human decision, as does drawing the area of interest and judging whether a
figure is plausible.